### 大规模分布式训练（Large-Scale Distributed Training）
以llama为例，这是相对开源程度高的大语言模型
#### GPU（Graphics Processing Unit）
GPU原本是为图像计算设计的，但由于在并行计算上表现突出，因此现在多被用于深度学习领域
<div align="center">
  <img src="class_images/inside_a_GPU.jpg" width="500">
</div>

H100 GPU 由很多个 SM 组成；每个 SM 里面有普通 FP32 Core 和专门做矩阵乘法的 Tensor Core。普通 FP32 Core 一次做标量乘加 a*x+b，而 Tensor Core 一次做小矩阵乘加 AX+B，所以在深度学习这种矩阵密集型任务中，Tensor Core 可以提供远高于普通浮点核心的计算吞吐量。

#### 如何在GPU上进行训练
* Data Parellelism：每张 GPU 都放一份完整模型，但处理不同的数据；最后把各 GPU 算出来的梯度取平均，再同步更新模型。但由于每个GPU都需要存储一个完整的模型，因此受制于显存，模型规模会受到限制
* Fully Sharded Data Parellelism：普通数据并行每张 GPU 都存完整模型，显存浪费严重；FSDP 把参数、梯度和优化器状态切分到不同 GPU 上，只在计算某一层时临时收集完整参数，因此大幅降低单张 GPU 的显存占用，使更大的模型可以训练
* Hybrid Sharded Data Parallel：它在一个 GPU 组内部用 FSDP 分片模型状态来省显存，在不同组之间像普通 DP 一样复制模型并同步梯度。它没有纯 FSDP 那么省显存，但能减少跨组/跨机器通信压力，更适合多机多卡大模型训练
* Activation Checkpointing：想法是在反向传播计算梯度时重新通过正向传播来计算所需层的结果，而不是在正向传播时把每一层的结果都存起来；这样做的好处是可以节省内存，但计算量增加，一个解决方法是每$C$层设置存档点，每次从存档点计算所需层的结果。为了平衡内存使用和计算量，通常取$C = \sqrt{N}$，其中$N$是模型层数
* Model FLOPs Utilization（MFU）：衡量训练模型时，实际用于模型计算的 FLOPs，占 GPU 理论峰值 FLOPs 的比例。这是调整相关超参数的依据
* 其它并行方法包括：Context Parellelism、Pipeline Parallelism、Tensor Parallelism